In [ ]:
# OMOJI INTENT CLASSIFIER - TRAINING & INFERENCE SCRIPT
# =====================================================================

import os
import logging
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer, 
    Trainer, 
    TrainingArguments,
    logging as hf_logging
)
import torch
import torch.nn as nn
import torch.nn.functional as F

# Enable verbose logging so VS Code displays download & training progress explicitly
hf_logging.set_verbosity_info()

# ---------------------------------------------------------------------
# 1. CATEGORY CONFIGURATION
# ---------------------------------------------------------------------
category_map = {0: "get", 1: "find", 2: "weigh"}
label_map_rev = {"get": 0, "find": 1, "weigh": 2}

texts = []
labels = []

file_path = "./training data/training-data.txt"

print(f"Reading training data from: {file_path}")
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and "|" in line:
            label_str, text = line.split("|", 1)
            label_str = label_str.strip().lower()
            if label_str in label_map_rev:
                texts.append(text.strip())
                labels.append(label_map_rev[label_str])

print(f"Successfully loaded {len(texts)} samples.")

dataset = Dataset.from_dict({"text": texts, "label": labels})

# ---------------------------------------------------------------------
# 2. MODEL & TOKENIZER SETUP
# ---------------------------------------------------------------------
model_name = "distilbert-base-uncased"

print(f"Loading tokenizer for '{model_name}'...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

print(f"Loading base model weights for '{model_name}'...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# ---------------------------------------------------------------------
# 3. LOGIC REINFORCEMENT: CUSTOM TRAINER WITH MARGIN/DECISIVENESS LOSS
# ---------------------------------------------------------------------
class DecisiveTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 1. Standard Cross Entropy Loss
        ce_loss = nn.CrossEntropyLoss()(logits, labels)
        
        # 2. Decisiveness Penalty: Penalize low-margin / low-confidence predictions during training
        probs = F.softmax(logits, dim=-1)
        top_probs, _ = torch.topk(probs, k=2, dim=-1)
        margin = top_probs[:, 0] - top_probs[:, 1]  # Gap between top 1 and top 2 class
        margin_penalty = torch.mean(1.0 - margin)   # Higher penalty when gap is small
        
        # Total loss combines cross entropy with margin enforcement
        total_loss = ce_loss + (0.2 * margin_penalty)
        
        return (total_loss, outputs) if return_outputs else total_loss

# ---------------------------------------------------------------------
# 4. TRAINING ARGUMENTS & EXECUTION (With Tweak Options & Comments)
# ---------------------------------------------------------------------

# --- OPTION 1: Your Current Baseline Setup ---
training_args = TrainingArguments(
    output_dir="./model_results",
    
    # Learning rate: 3e-5 is slightly aggressive, which helps the model 
    # push through the resistance introduced by your custom margin penalty.
    learning_rate=3e-5,            
    
    # Batch size: 4 works well for lightweight local execution, but keep 
    # in mind that small batches make margin calculations noisier.
    per_device_train_batch_size=4, 
    
    # Epochs: Set to 3 here, but TWEAK NOTE: if you are using the DecisiveTrainer 
    # margin penalty, bump this up to 8-10 epochs so the model has enough time 
    # to cleanly push class probabilities apart.
    num_train_epochs=3,            
    
    # Regularization: Standard weight decay to prevent overfitting on small data.
    weight_decay=0.01,
    
    # Logging: Set to 1 so you can inspect step-by-step loss interaction 
    # between CrossEntropy and your margin penalty.
    logging_steps=1                
)


# --- OPTION 2: Production & Scaling Tweak Example ---
# (Uncomment and use this block when you scale up your dataset or want smoother convergence)
# training_args = TrainingArguments(
#     output_dir="./model_results",
#     learning_rate=2e-5,              # Slightly safer, lower learning rate for stable long-run convergence
#     per_device_train_batch_size=8,   # Larger batch provides more stable top-2 probability gaps per step
#     gradient_accumulation_steps=2,   # Combines with batch size 8 for an effective batch size of 16
#     num_train_epochs=8,              # Optimal duration for margin-enforced separation
#     weight_decay=0.01,
#     warmup_ratio=0.1,                # Adds a learning rate warmup phase to prevent early erratic loss spikes
#     logging_steps=10                 # Reduces terminal clutter once your training loop is verified
# )

trainer = DecisiveTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
)

print("Starting training with decisiveness logic reinforcement...")
trainer.train()

# Save PyTorch model and tokenizer locally
save_directory = "./fine_tuned_classifier"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

# Rename default model.safetensors to 'half baked effort.safetensors'
old_safetensors_path = os.path.join(save_directory, "model.safetensors")
new_safetensors_path = os.path.join(save_directory, "pressure.safetensors")

if os.path.exists(old_safetensors_path):
    os.rename(old_safetensors_path, new_safetensors_path)
    print(f"Renamed safetensors weights to: '{new_safetensors_path}'")

print("Model saved successfully to './fine_tuned_classifier'!")

# ---------------------------------------------------------------------
# 5. INFERENCE & CONFIDENCE CLASSIFICATION LOGIC
# ---------------------------------------------------------------------
from IPython.display import display, HTML
display(HTML("<h3>🏁 Training Completed — Running Logic Validation...</h3><hr>"))

def evaluate_intent(prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
    outputs = model(**inputs)

    probabilities = F.softmax(outputs.logits, dim=1)
    confidence, predicted_class = torch.max(probabilities, dim=1)

    predicted_label = category_map[int(predicted_class.item())]
    confidence_score = confidence.item() * 100

    print(f"\n--- INFERENCE LOGIC TEST ---")
    print(f"Input: '{prompt}'")
    print(f"Top Predicted Intent: {predicted_label.upper()} ({confidence_score:.2f}% confidence)")

    # -----------------------------------------------------------------
    # ENFORCED BUSINESS LOGIC
    # -----------------------------------------------------------------
    HIGH_ACCURACY_THRESHOLD = 60.0
    DOUBT_THRESHOLD = 38.0

    if confidence_score >= HIGH_ACCURACY_THRESHOLD:
        status = "ACCURATE ENOUGH"
        action = f"Execute intent [{predicted_label.upper()}] directly."
    elif confidence_score <= DOUBT_THRESHOLD:
        status = "DOUBT"
        action = "High uncertainty / irrelevant input. Trigger fallback or ask user to rephrase."
    else:
        status = "AMBIGUOUS"
        action = f"Ask clarifying question: 'Did you mean to {predicted_label} this item?'"

    print(f"Classification Status: [{status}]")
    print(f"Omoji System Action: {action}\n")

# Run test prompt
test_text = "Kevin, you are the best motivational support guide for our work."
evaluate_intent(test_text)

In [ ]:
# Clean ONNX Export (Opset 18)
import os
import torch
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
from safetensors.torch import load_file

model_path = "./fine_tuned_classifier"
weights_file = "pressure.safetensors"  # Change this if using a different name

print("Loading tokenizer and model configuration...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
config = AutoConfig.from_pretrained(model_path)

# Initialize model from config (bypasses default filename checks)
model = AutoModelForSequenceClassification.from_config(config)

# Manually load your custom-named safetensors weights
weights_path = os.path.join(model_path, weights_file)
print(f"Loading weights from custom file: {weights_path}")
state_dict = load_file(weights_path)
model.load_state_dict(state_dict)
model.eval()

dummy_text = "sample query for model trace"
inputs = tokenizer(dummy_text, return_tensors="pt", max_length=128, padding="max_length", truncation=True)

onnx_output_path = "../../pizza-slize.onnx"

print("Exporting model to ONNX format...")
torch.onnx.export(
    model,
    (inputs["input_ids"], inputs["attention_mask"]),
    onnx_output_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=18,  # Match modern PyTorch default opset
    dynamo=False       # Use standard stable trace export
)

print(f"Successfully exported ONNX model to: {onnx_output_path}")